# Explainable Transfer Learning for Coral Reef Health Classification
**Student:** Sebastian | **Topic:** 5.4 | Machine Learning and Smart Systems

**Models:** Custom CNN · EfficientNetV2B0 · ConvNeXtTiny  
**Dataset:** BHD Corals (Healthy / Bleached / Dead)  
**Framework:** TensorFlow / Keras

## 1. Setup & Imports

In [ ]:
!pip install imagehash -q

In [ ]:
import os
import random
import shutil
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from PIL import Image
import imagehash

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2B0, ConvNeXtTiny
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, roc_auc_score, f1_score
)
from sklearn.calibration import calibration_curve

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Constants ─────────────────────────────────────────────────────────────────
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 3
CLASS_NAMES = ['Healthy', 'Bleached', 'Dead']
EPOCHS_SCRATCH   = 50
EPOCHS_FINETUNE  = 30
HASH_THRESHOLD   = 8   # perceptual hash hamming distance for near-duplicates

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

## 2. Dataset Paths (BHD Corals on Kaggle)

In [ ]:
# ── Kaggle input path ─────────────────────────────────────────────────────────
# Dataset: https://www.kaggle.com/datasets/bouweceunen/bhd-corals
# Add via: Notebook Settings → Add Data → search 'BHD Corals'
DATA_ROOT = Path('/kaggle/input/datasets/sonainjamil/bhd-corals/Dataset')

# Auto-detect subdirectory layout
def find_class_dirs(root):
    """Find directories named healthy/bleached/dead anywhere under root."""
    for p in sorted(root.rglob('*')):
        if p.is_dir() and p.name in CLASS_NAMES:
            return p.parent
    return root

DATASET_DIR = find_class_dirs(DATA_ROOT)
print(f'Dataset directory: {DATASET_DIR}')

for cls in CLASS_NAMES:
    cls_path = DATASET_DIR / cls
    if cls_path.exists():
        n = len(list(cls_path.glob('*.*')))
        print(f'  {cls}: {n} images')
    else:
        # Try case-insensitive match
        matches = [d for d in DATASET_DIR.iterdir() if d.name.lower() == cls]
        if matches:
            n = len(list(matches[0].glob('*.*')))
            print(f'  {cls} (found as {matches[0].name}): {n} images')
        else:
            print(f'  WARNING: class "{cls}" not found under {DATASET_DIR}')

## 3. Dataset Audit & Deduplication

In [ ]:
def collect_image_paths(dataset_dir, class_names):
    """Return DataFrame with columns: path, label."""
    records = []
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    for cls in class_names:
        cls_dir = dataset_dir / cls
        if not cls_dir.exists():
            # try case-insensitive
            candidates = [d for d in dataset_dir.iterdir() if d.name == cls]
            cls_dir = candidates[0] if candidates else None
        if cls_dir and cls_dir.exists():
            for p in cls_dir.iterdir():
                if p.suffix.lower() in exts:
                    records.append({'path': str(p), 'label': cls})
    return pd.DataFrame(records)

df_all = collect_image_paths(DATASET_DIR, CLASS_NAMES)
print(f'Total images found: {len(df_all)}')
print(df_all['label'].value_counts())

In [ ]:
def compute_phash(path):
    try:
        return str(imagehash.phash(Image.open(path).convert('RGB')))
    except Exception:
        return None

print('Computing perceptual hashes (may take a few minutes)...')
df_all['phash'] = df_all['path'].apply(compute_phash)

# Remove exact-hash duplicates (keep first)
before = len(df_all)
df_all = df_all.dropna(subset=['phash'])
df_all = df_all.drop_duplicates(subset='phash', keep='first').reset_index(drop=True)
after = len(df_all)
print(f'Removed {before - after} duplicates → {after} unique images')
print(df_all['label'].value_counts())

## 4. Stratified 70 / 15 / 15 Split

In [ ]:
# 70 train | 30 temp → 50/50 of temp = 15 val + 15 test
df_train, df_temp = train_test_split(
    df_all, test_size=0.30, stratify=df_all['label'], random_state=SEED
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50, stratify=df_temp['label'], random_state=SEED
)

for name, df in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    print(f'{name} ({len(df)}): {dict(df["label"].value_counts())}')

In [ ]:
# ── Stage images into temp dirs so Keras ImageDataGenerator can read them ────
STAGE_DIR = Path('/kaggle/working/staged')

def stage_split(df, split_name):
    for _, row in df.iterrows():
        dst = STAGE_DIR / split_name / row['label']
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy2(row['path'], dst / Path(row['path']).name)

if not STAGE_DIR.exists():
    print('Staging files...')
    stage_split(df_train, 'train')
    stage_split(df_val,   'val')
    stage_split(df_test,  'test')
    print('Done.')
else:
    print('Stage directory already exists, skipping.')

## 5. Data Generators & Class Weights

In [ ]:
# ── Augmentation (train only) ─────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    channel_shift_range=30.0,   # simulates underwater colour shift
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    STAGE_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True,
    seed=SEED
)

val_gen = val_test_datagen.flow_from_directory(
    STAGE_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    STAGE_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print('Class indices:', train_gen.class_indices)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

y_train_labels = df_train['label'].values
cw_values = compute_class_weight(
    class_weight='balanced',
    classes=np.array(CLASS_NAMES),  # fix: numpy array
    y=y_train_labels
)
class_weight_dict = {i: w for i, w in enumerate(cw_values)}
print('Class weights:', class_weight_dict)

## 6. Model Definitions

In [ ]:
# ── Shared callbacks factory ──────────────────────────────────────────────────
def get_callbacks(model_name):
    return [
        EarlyStopping(
            monitor='val_loss', patience=8,
            restore_best_weights=True, verbose=1
        ),
        ModelCheckpoint(
            f'/kaggle/working/{model_name}_best.keras',
            monitor='val_loss', save_best_only=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4,
            min_lr=1e-6, verbose=1
        )
    ]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# MODEL 1: Custom CNN (trained from scratch)
# ────────────────────────────────────────────────────────────────────────────
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=3):
    inputs = keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Block 4
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.25)(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='CustomCNN')
    return model

cnn_model = build_custom_cnn()
cnn_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# MODEL 2: EfficientNetV2B0 (transfer learning)
# ────────────────────────────────────────────────────────────────────────────
def build_efficientnetv2(num_classes=3, input_shape=(224, 224, 3)):
    base = EfficientNetV2B0(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    # Phase 1: freeze base
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    # EfficientNetV2 expects uint8 [0,255] internally — re-scale back
    x = layers.Rescaling(255.0)(inputs)   # undo the /255 from generator
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='EfficientNetV2B0')
    return model, base

effnet_model, effnet_base = build_efficientnetv2()
effnet_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
effnet_model.summary()

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# MODEL 3: ConvNeXtTiny (transfer learning)
# ────────────────────────────────────────────────────────────────────────────
def build_convnext(num_classes=3, input_shape=(224, 224, 3)):
    base = ConvNeXtTiny(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(255.0)(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='ConvNeXtTiny')
    return model, base

convnext_model, convnext_base = build_convnext()
convnext_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
convnext_model.summary()

## 7. Training

In [ ]:
# ── 7.1 Custom CNN (from scratch) ────────────────────────────────────────────
print('=' * 60)
print('Training Custom CNN...')
print('=' * 60)

t0 = time.time()
cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_SCRATCH,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('cnn'),
    verbose=1
)
cnn_train_time = time.time() - t0
print(f'Custom CNN training time: {cnn_train_time:.1f}s')

In [ ]:
# ── 7.2 EfficientNetV2 — Phase 1: frozen base ─────────────────────────────────
print('=' * 60)
print('Training EfficientNetV2B0 — Phase 1 (head only)...')
print('=' * 60)

t0 = time.time()
eff_hist1 = effnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('effnet_phase1'),
    verbose=1
)

# Phase 2: fine-tune top layers of the base
print('\nPhase 2: Fine-tuning top 30 layers of EfficientNetV2B0...')
effnet_base.trainable = True
for layer in effnet_base.layers[:-30]:
    layer.trainable = False

effnet_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

eff_hist2 = effnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('effnet_phase2'),
    verbose=1
)
effnet_train_time = time.time() - t0
print(f'EfficientNetV2 total training time: {effnet_train_time:.1f}s')

In [ ]:
# ── 7.3 ConvNeXtTiny — Phase 1 + Fine-tune ────────────────────────────────────
print('=' * 60)
print('Training ConvNeXtTiny — Phase 1 (head only)...')
print('=' * 60)

t0 = time.time()
cnx_hist1 = convnext_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('convnext_phase1'),
    verbose=1
)

print('\nPhase 2: Fine-tuning top 30 layers of ConvNeXtTiny...')
convnext_base.trainable = True
for layer in convnext_base.layers[:-30]:
    layer.trainable = False

convnext_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnx_hist2 = convnext_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINETUNE,
    class_weight=class_weight_dict,
    callbacks=get_callbacks('convnext_phase2'),
    verbose=1
)
convnext_train_time = time.time() - t0
print(f'ConvNeXtTiny total training time: {convnext_train_time:.1f}s')

## 8. Evaluation & Metrics

In [ ]:
def evaluate_model(model, gen, model_name, train_time):
    """Returns a dict with all required metrics."""
    gen.reset()

    # Inference timing
    t0 = time.time()
    y_prob = model.predict(gen, verbose=0)
    inference_time_ms = (time.time() - t0) / len(gen.filenames) * 1000

    y_pred = np.argmax(y_prob, axis=1)
    y_true = gen.classes

    # Core metrics
    acc          = np.mean(y_pred == y_true)
    bal_acc      = balanced_accuracy_score(y_true, y_pred)
    macro_f1     = f1_score(y_true, y_pred, average='macro')
    per_cls_f1   = f1_score(y_true, y_pred, average=None)

    # One-vs-rest ROC-AUC
    roc_auc = roc_auc_score(
        tf.keras.utils.to_categorical(y_true, NUM_CLASSES),
        y_prob, average='macro', multi_class='ovr'
    )

    # Expected Calibration Error (ECE)
    confidences = np.max(y_prob, axis=1)
    correctness = (y_pred == y_true).astype(float)
    n_bins = 15
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bin_edges[i]) & (confidences <= bin_edges[i+1])
        if mask.sum() > 0:
            ece += mask.sum() * abs(correctness[mask].mean() - confidences[mask].mean())
    ece /= len(y_true)

    # Model size & params
    model.save(f'/kaggle/working/{model_name}_final.keras')
    model_size_mb = os.path.getsize(f'/kaggle/working/{model_name}_final.keras') / 1e6
    param_count = model.count_params()

    cm_matrix = confusion_matrix(y_true, y_pred)

    print(f'\n{"="*55}')
    print(f'  {model_name}')
    print(f'{"="*55}')
    print(f'  Accuracy:         {acc:.4f}')
    print(f'  Balanced Acc:     {bal_acc:.4f}')
    print(f'  Macro F1:         {macro_f1:.4f}')
    print(f'  ROC-AUC (macro):  {roc_auc:.4f}')
    print(f'  ECE:              {ece:.4f}')
    print(f'  Train time:       {train_time:.1f}s')
    print(f'  Inference/img:    {inference_time_ms:.2f} ms')
    print(f'  Parameters:       {param_count:,}')
    print(f'  Model size:       {model_size_mb:.1f} MB')
    print(f'  Per-class F1:     {dict(zip(CLASS_NAMES, per_cls_f1))}')
    print(f'\n{classification_report(y_true, y_pred, target_names=CLASS_NAMES)}')

    return {
        'name': model_name,
        'accuracy': acc, 'balanced_accuracy': bal_acc,
        'macro_f1': macro_f1, 'per_class_f1': per_cls_f1,
        'roc_auc': roc_auc, 'ece': ece,
        'train_time': train_time, 'inference_ms': inference_time_ms,
        'params': param_count, 'size_mb': model_size_mb,
        'confusion_matrix': cm_matrix,
        'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred
    }

test_gen.reset()
results = {}
results['cnn']       = evaluate_model(cnn_model,      test_gen, 'CustomCNN',      cnn_train_time)
results['effnet']    = evaluate_model(effnet_model,    test_gen, 'EfficientNetV2', effnet_train_time)
results['convnext']  = evaluate_model(convnext_model,  test_gen, 'ConvNeXtTiny',   convnext_train_time)

## 9. Visualization: Confusion Matrices & Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (key, res) in zip(axes, results.items()):
    cm_norm = res['confusion_matrix'].astype('float') / \
              res['confusion_matrix'].sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', ax=ax, vmin=0, vmax=1
    )
    ax.set_title(f'{res["name"]}\nMacro F1={res["macro_f1"]:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Normalized Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# Comparison bar chart
metrics_df = pd.DataFrame([
    {
        'Model': r['name'],
        'Accuracy': r['accuracy'],
        'Balanced Acc': r['balanced_accuracy'],
        'Macro F1': r['macro_f1'],
        'ROC-AUC': r['roc_auc'],
        'ECE': r['ece'],
        'Inference (ms)': r['inference_ms'],
        'Params (M)': r['params'] / 1e6,
        'Size (MB)': r['size_mb']
    }
    for r in results.values()
])

print('\n── Model Comparison Table ──')
display(metrics_df.set_index('Model').round(4))
metrics_df.to_csv('/kaggle/working/model_comparison.csv', index=False)

# Bar chart
plot_cols = ['Accuracy', 'Balanced Acc', 'Macro F1', 'ROC-AUC']
metrics_df.set_index('Model')[plot_cols].plot(
    kind='bar', figsize=(10, 5), ylim=(0, 1.05),
    rot=0, colormap='viridis'
)
plt.title('Model Performance Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('/kaggle/working/model_comparison.png', dpi=150)
plt.show()

## 10. Grad-CAM Explainability

In [ ]:
def grad_cam_transfer(model, img_array, class_idx, base_name):
    """Grad-CAM para modelos con base preentrenada (EfficientNet, ConvNeXt)."""
    base = model.get_layer(base_name)
    
    # Encontrar la última conv dentro del base
    last_conv = None
    for layer in reversed(base.layers):
        if isinstance(layer, layers.Conv2D):
            last_conv = layer.name
            break
    
    if last_conv is None:
        # ConvNeXt no usa Conv2D estándar, usar la última capa antes del pooling
        for layer in reversed(base.layers):
            if 'layer_norm' in layer.name.lower() or 'norm' in layer.name.lower():
                last_conv = layer.name
                break
    
    print(f'  Usando capa: {last_conv}')
    
    grad_model = tf.keras.Model(
        inputs=base.input,
        outputs=[base.get_layer(last_conv).output, base.output]
    )
    
    # Pasar imagen por el rescaling manualmente (x255)
    img_rescaled = img_array * 255.0
    img_tensor = tf.cast(img_rescaled[np.newaxis, ...], tf.float32)
    
    with tf.GradientTape() as tape:
        conv_outputs, features = grad_model(img_tensor)
        # Obtener predicción del modelo completo para el class_idx
        full_pred = model(img_array[np.newaxis].astype('float32'))
        loss = full_pred[:, class_idx]
    
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        return np.zeros((7, 7))
    grads = grads[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap).numpy()
    if heatmap.ndim == 0:
        return np.zeros((7, 7))
    heatmap = np.maximum(heatmap, 0) / (heatmap.max() + 1e-8)
    return heatmap


def show_gradcam_grid_v2(model, model_name, gen, n_per_class=2):
    gen.reset()
    
    # Detectar si es transfer learning o CNN custom
    base_name = None
    for layer in model.layers:
        if hasattr(layer, 'layers') and len(layer.layers) > 5:
            base_name = layer.name
            break
    
    print(f'{model_name}: base = {base_name}')
    
    # Recolectar muestras
    samples = {c: [] for c in range(NUM_CLASSES)}
    for batch_imgs, batch_lbls in gen:
        for img, lbl in zip(batch_imgs, batch_lbls):
            c = np.argmax(lbl)
            if len(samples[c]) < n_per_class:
                samples[c].append(img)
        if all(len(v) >= n_per_class for v in samples.values()):
            break

    fig, axes = plt.subplots(
        NUM_CLASSES * 2, n_per_class + 1,
        figsize=(5 * (n_per_class + 1), 4 * NUM_CLASSES)
    )

    for row_base, (cls_idx, imgs) in enumerate(samples.items()):
        row_img = row_base * 2
        row_cam = row_base * 2 + 1
        cls_name = CLASS_NAMES[cls_idx]

        axes[row_img, 0].text(0.5, 0.5, cls_name, ha='center', va='center',
                               fontsize=13, fontweight='bold')
        axes[row_img, 0].axis('off')
        axes[row_cam, 0].axis('off')

        for col, img in enumerate(imgs[:n_per_class]):
            probs  = model.predict(img[np.newaxis], verbose=0)[0]
            pred_c = np.argmax(probs)
            
            if base_name:
                heatmap = grad_cam_transfer(model, img, pred_c, base_name)
            else:
                layer_info = get_last_conv_layer(model)
                heatmap = grad_cam(model, img, pred_c, layer_info)
            
            overlay = overlay_gradcam(img, heatmap)

            axes[row_img, col + 1].imshow(img)
            axes[row_img, col + 1].set_title(
                f'Pred: {CLASS_NAMES[pred_c]} ({probs[pred_c]:.2f})', fontsize=9)
            axes[row_img, col + 1].axis('off')

            axes[row_cam, col + 1].imshow(overlay)
            axes[row_cam, col + 1].set_title('Grad-CAM', fontsize=9)
            axes[row_cam, col + 1].axis('off')

    plt.suptitle(f'Grad-CAM — {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/gradcam_{model_name}.png', dpi=150)
    plt.show()


show_gradcam_grid_v2(cnn_model,      'CustomCNN',      test_gen)
show_gradcam_grid_v2(effnet_model,   'EfficientNetV2', test_gen)
show_gradcam_grid_v2(convnext_model, 'ConvNeXtTiny',   test_gen)

## 11. Colour Ablation Experiment (RQ4)

In [ ]:
def colour_ablation(model, model_name, gen, n_samples=200):
    """
    Compare accuracy on:
      (a) original RGB
      (b) grayscale (repeat across 3 channels)
      (c) colour-jittered (strong channel shift)
    """
    gen.reset()
    imgs, labels = [], []
    for batch_imgs, batch_lbls in gen:
        imgs.append(batch_imgs)
        labels.append(batch_lbls)
        if sum(len(x) for x in imgs) >= n_samples:
            break
    imgs   = np.concatenate(imgs)[:n_samples]
    labels = np.concatenate(labels)[:n_samples]
    y_true = np.argmax(labels, axis=1)

    def acc(transformed):
        preds = np.argmax(model.predict(transformed, verbose=0, batch_size=32), axis=1)
        return np.mean(preds == y_true)

    # (a) Original
    acc_orig = acc(imgs)

    # (b) Grayscale
    gray = np.mean(imgs, axis=-1, keepdims=True)
    gray_rgb = np.repeat(gray, 3, axis=-1)
    acc_gray = acc(gray_rgb)

    # (c) Colour jitter (swap R and B channels — simulates blue underwater shift)
    jitter = imgs.copy()
    jitter[..., 0] = np.clip(imgs[..., 0] * 0.5, 0, 1)  # reduce red
    jitter[..., 2] = np.clip(imgs[..., 2] * 1.5, 0, 1)  # boost blue
    acc_jitter = acc(jitter)

    print(f'\n── Colour Ablation: {model_name} ──')
    print(f'  Original RGB:    {acc_orig:.4f}')
    print(f'  Grayscale:       {acc_gray:.4f}  (Δ={acc_gray - acc_orig:+.4f})')
    print(f'  Colour-jittered: {acc_jitter:.4f}  (Δ={acc_jitter - acc_orig:+.4f})')

    return {'model': model_name,
            'acc_rgb': acc_orig, 'acc_gray': acc_gray, 'acc_jitter': acc_jitter}


ablation_results = [
    colour_ablation(cnn_model,      'CustomCNN',      test_gen),
    colour_ablation(effnet_model,   'EfficientNetV2', test_gen),
    colour_ablation(convnext_model, 'ConvNeXtTiny',   test_gen),
]

# Plot
abl_df = pd.DataFrame(ablation_results).set_index('model')
abl_df.columns = ['RGB', 'Grayscale', 'Colour-Jitter']
abl_df.plot(kind='bar', figsize=(9, 5), rot=0, ylim=(0, 1))
plt.title('Colour Ablation — Accuracy by Model and Input Type')
plt.ylabel('Accuracy')
plt.legend(title='Input')
plt.tight_layout()
plt.savefig('/kaggle/working/colour_ablation.png', dpi=150)
plt.show()

## 12. Calibration (Expected Calibration Error)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (key, res) in zip(axes, results.items()):
    y_true_bin = (res['y_pred'] == res['y_true']).astype(int)
    confidences = np.max(res['y_prob'], axis=1)

    fraction_of_pos, mean_pred = calibration_curve(
        y_true_bin, confidences, n_bins=10, strategy='uniform'
    )
    ax.plot(mean_pred, fraction_of_pos, 's-', label=res['name'])
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect')
    ax.set_title(f"{res['name']}\nECE={res['ece']:.4f}")
    ax.set_xlabel('Mean Predicted Confidence')
    ax.set_ylabel('Fraction Correct')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.suptitle('Reliability / Calibration Diagrams', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/calibration.png', dpi=150)
plt.show()

## 13. Learning Curves

In [ ]:
def plot_history(history, title, ax_loss, ax_acc):
    epochs = range(1, len(history.history['loss']) + 1)
    ax_loss.plot(epochs, history.history['loss'], label='Train')
    ax_loss.plot(epochs, history.history['val_loss'], label='Val')
    ax_loss.set_title(f'{title} — Loss')
    ax_loss.set_xlabel('Epoch')
    ax_loss.legend()

    ax_acc.plot(epochs, history.history['accuracy'], label='Train')
    ax_acc.plot(epochs, history.history['val_accuracy'], label='Val')
    ax_acc.set_title(f'{title} — Accuracy')
    ax_acc.set_xlabel('Epoch')
    ax_acc.legend()

fig, axes = plt.subplots(3, 2, figsize=(14, 14))

plot_history(cnn_history,   'Custom CNN',      axes[0, 0], axes[0, 1])
plot_history(eff_hist2,     'EfficientNetV2',  axes[1, 0], axes[1, 1])
plot_history(cnx_hist2,     'ConvNeXtTiny',    axes[2, 0], axes[2, 1])

plt.suptitle('Learning Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=150)
plt.show()

## 14. Summary Table

In [ ]:
summary = pd.DataFrame([
    {
        'Model':            r['name'],
        'Accuracy':         round(r['accuracy'], 4),
        'Balanced Acc':     round(r['balanced_accuracy'], 4),
        'Macro F1':         round(r['macro_f1'], 4),
        'ROC-AUC':          round(r['roc_auc'], 4),
        'ECE (↓)':          round(r['ece'], 4),
        'F1 Healthy':       round(r['per_class_f1'][0], 4),
        'F1 Bleached':      round(r['per_class_f1'][1], 4),
        'F1 Dead':          round(r['per_class_f1'][2], 4),
        'Train Time (s)':   round(r['train_time'], 1),
        'Inference (ms)':   round(r['inference_ms'], 2),
        'Params (M)':       round(r['params'] / 1e6, 2),
        'Size (MB)':        round(r['size_mb'], 1)
    }
    for r in results.values()
])

display(summary.set_index('Model'))
summary.to_csv('/kaggle/working/final_summary.csv', index=False)
print('\nAll outputs saved to /kaggle/working/')